# Episode 22: Make Agents Think Harder: Tree of Thoughts

Standard LLMs answer in one shot. For hard problems, that's often not enough. What if your agent could explore multiple reasoning paths and pick the best one?

In this episode, you'll learn how to use structured reasoning strategies — beam search, DFS, MCTS, and LATS — to dramatically improve agent problem-solving.

---

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Theory: Why Standard LLMs Struggle with Complex Problems

Standard LLMs generate answers in **one pass** — tokens flow left-to-right without backtracking. For simple questions, this works fine. For complex problems, it's like trying to solve a maze by always turning right.

The idea that intelligence emerges from structured collaboration between simpler processes isn't new. Marvin Minsky argued in *The Society of Mind* (1986) that intelligence arises from simple agents cooperating within structured frameworks. **Tree of Thoughts (ToT)** applies this principle to LLM reasoning: instead of one monolithic pass, the agent creates a tree of reasoning paths and evaluates which branch is most promising.

```
Problem
├── Approach A
│   ├── Step A1 → Score: 7/10 ✓ (expand)
│   └── Step A2 → Score: 3/10 ✗ (prune)
├── Approach B
│   ├── Step B1 → Score: 8/10 ✓ (expand)
│   └── Step B2 → Score: 6/10 ✓ (expand)
└── Approach C
    └── Step C1 → Score: 2/10 ✗ (prune)
```

The agent **evaluates** each step before continuing, pruning bad paths early and focusing compute on promising ones.

## Search Strategies

AG2's `ReasoningAgent` supports four search strategies, each suited to different problem types:

| Strategy | How it works | Best for |
|----------|-------------|----------|
| **Beam Search** (default) | Keep top-k most promising paths, expand them all | Balanced exploration when you can afford it |
| **DFS** (`beam_size=1`) | Follow one path deeply, similar to chain-of-thought | Fast, linear reasoning (O1-style) |
| **MCTS** | Monte Carlo Tree Search — balance exploration vs exploitation | Game-like problems with clear win/lose states |
| **LATS** | Language Agent Tree Search — adds reflection feedback loops | Complex multi-step reasoning with self-correction |

Start with **beam search** for quality, switch to **DFS** when speed matters more than thoroughness.

## Beam Search: Explore Multiple Paths

Beam search keeps the top-k most promising reasoning paths at each step, expanding all of them before pruning again. Let's see it in action on a probability problem.

In [ ]:
from autogen import ConversableAgent, LLMConfig
from autogen.agents.experimental import ReasoningAgent

llm_config = LLMConfig({"model": "gpt-4o-mini"})

reasoning_agent = ReasoningAgent(
    name="thinker",
    reason_config={
        "method": "beam_search",
        "max_depth": 3,
        "beam_size": 2,
    },
)
user = ConversableAgent("user", llm_config=llm_config, human_input_mode="NEVER")

result = user.initiate_chat(
    reasoning_agent,
    message="What is the expected maximum value when rolling a 6-sided die three times?",
    max_turns=1,
)
print(result.summary)

Notice the agent explored multiple reasoning paths — it considered different approaches (combinatorial, probabilistic, simulation-based) and selected the most promising one.

With `beam_size=2`, it kept the top 2 paths at each step. More beams means more exploration, better answers, and higher cost. It's a dial you can tune.

## DFS: Fast Linear Reasoning

When speed matters more than exhaustive exploration, DFS follows a single reasoning path deeply — similar to how O1-style models think step by step along one chain.

In [ ]:
dfs_agent = ReasoningAgent(
    name="fast_thinker",
    reason_config={
        "method": "dfs",
        "max_depth": 3,
    },
)
user2 = ConversableAgent("user", llm_config=llm_config, human_input_mode="NEVER")

result_dfs = user2.initiate_chat(
    dfs_agent,
    message="What is the expected maximum value when rolling a 6-sided die three times?",
    max_turns=1,
)
print(result_dfs.summary)

**How do they compare?** Beam search explored multiple paths and picked the best, making it more thorough but more expensive. DFS followed one path deeply — faster and cheaper, but it might miss better approaches.

The right choice depends on your problem. For high-stakes decisions, beam search is worth the extra cost. For routine queries, DFS gives you structured reasoning at near-standard speed.

## Advanced: MCTS and LATS

### Monte Carlo Tree Search (MCTS)

MCTS balances exploration (trying new paths) against exploitation (deepening promising ones). It's particularly effective for problems where you can clearly evaluate intermediate states.

```python
reason_config={
    "method": "mcts",
    "max_depth": 4,
    "nsim": 3,  # number of simulations per node
}
```

### Language Agent Tree Search (LATS)

LATS extends MCTS with **reflection** — the agent reviews failed paths and feeds that insight back into future attempts. This self-correction loop makes it the most powerful strategy, but also the most expensive.

```python
reason_config={
    "method": "lats",
    "max_depth": 4,
    "nsim": 3,
}
```

## Training Data Extraction

A practical bonus of tree-based reasoning: you can extract the reasoning trees as **training data** to fine-tune smaller models.

With **SFT (Supervised Fine-Tuning)**, you extract successful reasoning paths as input-output pairs. With **RLHF**, you use the scored paths as preference data — high-scored paths are preferred over low-scored ones.

This is how you use a large model (GPT-4o) to generate reasoning data that trains a smaller, cheaper model to reason similarly. It's distillation with structure.

## Try It Yourself

1. Change `beam_size` to 3 in the beam search example. Does the answer quality improve? Check the token usage to see the cost tradeoff.

2. Try a harder problem:
   ```python
   message = "Design a system to detect fraudulent transactions in real-time for an e-commerce platform processing 10,000 transactions per second."
   ```
   Compare beam search vs DFS — which gives a more thorough answer?

3. Experiment with `max_depth`. Deeper reasoning isn't always better — find the sweet spot for your problem.

## Reference

- AG2 blog: [ReasoningAgent](https://docs.ag2.ai/blog/2024-12-02-ReasoningAgent2)
- AG2 blog: [Reasoning Update](https://docs.ag2.ai/blog/2024-12-20-Reasoning-Update)
- Minsky, *The Society of Mind*, 1986
- Original Tree of Thoughts paper: Yao et al., 2023

---

**What's Next:** The final episode — what's coming in AG2's future.